# Masked Auto-Encoding ([MAE](https://arxiv.org/abs/2111.06377)) Demo

This notebook is a compact, end-to-end walkthrough of MAE on Imagenette. You will prepare data, load a pretrained MAE model, inspect reconstructions, and run a short pretraining loop to see how reconstruction quality evolves.


<img src="https://user-images.githubusercontent.com/11435359/146857310-f258c86c-fde6-48e8-9cee-badd2b21bd2c.png" width="800">


## 0) Import Libraries & Install Dependencies

We start by importing standard Python tools and defining small utilities for shell commands and downloads. The setup is written to run cleanly in a regular Jupyter environment or in Colab.


### Basic imports
Libraries that are already installed in most Python environments.

This section keeps core utilities together so the rest of the notebook can focus on MAE and data workflow.


In [ ]:
from pathlib import Path
import random
import shlex
import subprocess
import sys
import re
import urllib.request
import tarfile
import os

### Useful Helpers

These helper functions keep later cells short and readable:
- `run_cmd(...)` runs shell commands consistently.
- environment checks keep behavior stable in notebook vs Colab.
- download/export helpers make dataset preparation repeatable.


In [ ]:
# run a command, optionally in a specific directory, and optionally checking for errors
def run_cmd(cmd, *, cwd=None, check=True):
    if isinstance(cmd, str):
        return subprocess.run(cmd, shell=True, cwd=cwd, check=check)
    return subprocess.run([str(c) for c in cmd], cwd=cwd, check=check)

# check whether we're running in a Jupyter notebook or Google Colab
def is_colab() -> bool:
    return "google.colab" in sys.modules 

# check if directory contains any .jpg files
def has_jpgs(path: Path) -> bool:
    return path.exists() and any(path.rglob('*.jpg'))


def split_export_done(path: Path) -> bool:
    return (path / '.export_done').exists() and has_jpgs(path)

def download_with_progress(url: str, dst: Path):
    from tqdm.auto import tqdm
    if dst.exists():
        print(f'Imagenette archive already exists: {dst}')
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    print(f'Downloading Imagenette archive from {url}')
    with urllib.request.urlopen(url) as response, open(dst, 'wb') as f:
        total = int(response.headers.get('Content-Length', 0))
        with tqdm(total=total if total > 0 else None, unit='B', unit_scale=True, desc='Imagenette download') as pbar:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                f.write(chunk)
                pbar.update(len(chunk))

# Save JPEG with Pillow
def save_jpeg_robust(pil_img, out_file: Path, quality: int = 95):
    import numpy as np
    out_file = Path(out_file)
    img = pil_img if isinstance(pil_img, Image.Image) else Image.fromarray(np.asarray(pil_img))
    img.save(out_file, format='JPEG', quality=int(quality))



### Install and import other dependencies if missing

Install the modeling and visualization packages used in this notebook, then import them once for all later sections.


In [ ]:
required_packages = ('torch', 
                     'torchvision', 
                     'timm', 
                     'tensorboard', 
                     'numpy', 
                     'tqdm', 
                     'matplotlib',
                     'Pillow',)
run_cmd([sys.executable, '-m', 'pip', 'install', '--quiet', *required_packages])

from tqdm.auto import tqdm
import numpy as np
from PIL import Image
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

## 1) Create Workspace Folders

Define a small, reproducible project layout (`data/`, `outputs/`, `checkpoints/`, and local MAE repo folder). Running this cell multiple times is safe.


In [ ]:
ROOT = Path('.').resolve()
MAE_REPO_DIR = ROOT / 'mae'

DATA_ROOT = ROOT / 'data'
IMAGENETTE_DIR = DATA_ROOT / 'imagenette'
PETS_DIR = DATA_ROOT / 'pets'
OUTPUTS_DIR = ROOT / 'outputs'
RECON_DIR = OUTPUTS_DIR / 'recon'
CHECKPOINTS_DIR = ROOT / 'checkpoints'

for p in [
    DATA_ROOT,
    IMAGENETTE_DIR / 'train',
    IMAGENETTE_DIR / 'val',
    PETS_DIR / 'train',
    PETS_DIR / 'val',
    OUTPUTS_DIR,
    RECON_DIR,
    CHECKPOINTS_DIR,
    MAE_REPO_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

print('Workspace folders are ready.')


## 2) Clone [MAE](https://github.com/facebookresearch/mae) Repo

Clone the official MAE repository, then apply a few tiny compatibility patches for modern NumPy/Timm/PyTorch versions. The patching step is idempotent, so reruns remain clean.


In [ ]:
# patch a text file, used to fix some deprecation issues in MAE
def patch_text_file(path: Path, transform, label: str):
    if not path.exists():
        print(f'[missing] {label}: {path}')
        return False
    old = path.read_text()
    new = transform(old)
    if new != old:
        path.write_text(new)
        print(f'[patched] {label}: {path}')
        return True
    print(f'[ok] {label}: {path}')
    return False

if MAE_REPO_DIR.exists() and (MAE_REPO_DIR / '.git').exists():
    print(f'MAE repo already exists at {MAE_REPO_DIR}, skipping clone.')
else:
    cmd = ['git', 'clone', 'https://github.com/facebookresearch/mae.git', str(MAE_REPO_DIR)]
    print('Running:', ' '.join(cmd))
    run_cmd(cmd, check=True)


pos_embed_path = MAE_REPO_DIR / 'util' / 'pos_embed.py'
models_mae_path = MAE_REPO_DIR / 'models_mae.py'
misc_path = MAE_REPO_DIR / 'util' / 'misc.py'

# numpy compatibility: replace deprecated np.float alias
patch_text_file(
    pos_embed_path,
    lambda t: re.sub(r'np\.float(?!\d)', 'np.float32', t),
    'replace deprecated np.float with np.float32',
)
# timm compatibility: newer Block no longer supports qk_scale arg
patch_text_file(
    models_mae_path,
    lambda t: re.sub(r',\s*qk_scale\s*=\s*None', '', t),
    'remove qk_scale=None from Block(...) calls',
)
# torch compatibility: torch._six removed in modern torch
patch_text_file(
    misc_path,
    lambda t: t.replace('from torch._six import inf', 'from math import inf'),
    'replace torch._six import inf',
)

# Quick sanity checks
legacy_np_float = bool(re.search(r'np\.float(?!\d)', pos_embed_path.read_text()))
legacy_qk_scale = 'qk_scale' in models_mae_path.read_text()
legacy_torch_six = 'from torch._six import inf' in misc_path.read_text()

if legacy_np_float or legacy_qk_scale or legacy_torch_six:
    print('WARNING: some compatibility patterns remain after patching:')
    print(f'  np.float present: {legacy_np_float}')
    print(f'  qk_scale present: {legacy_qk_scale}')
    print(f'  torch._six present: {legacy_torch_six}')
else:
    print('MAE compatibility patch status: OK')



## 3) Export Imagenette to ImageFolder

Download and extract Imagenette, then export it into a clean `ImageFolder`-style train/val datasets. 


In [ ]:
IMAGENETTE_URL = 'https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz'
IMAGENETTE_ARCHIVE_PATH = DATA_ROOT / 'imagenette2-320.tgz'
IMAGENETTE_EXTRACT_DIR = DATA_ROOT / 'imagenette2-320'

imagenette_train_dir = IMAGENETTE_DIR / 'train'
imagenette_val_dir = IMAGENETTE_DIR / 'val'

if IMAGENETTE_ARCHIVE_PATH.exists() and IMAGENETTE_EXTRACT_DIR.exists():
    print(f'Imagenette archive and extracted data already exist, skipping download and extraction.')

download_with_progress(IMAGENETTE_URL, IMAGENETTE_ARCHIVE_PATH)
print(f'Extracting {IMAGENETTE_ARCHIVE_PATH} -> {DATA_ROOT}')
with tarfile.open(IMAGENETTE_ARCHIVE_PATH, 'r:gz') as tar:
    tar.extractall(path=DATA_ROOT)

split_map = {
    'train': IMAGENETTE_EXTRACT_DIR / 'train',
    'val': IMAGENETTE_EXTRACT_DIR / 'val',
}
valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

for split_name, src_split_dir in split_map.items():
    dst_split_dir = IMAGENETTE_DIR / split_name
    marker_file = dst_split_dir / '.export_done'

    if split_export_done(dst_split_dir):
        print(f'Imagenette {split_name}: already exported, skipping.')
        continue

    if not src_split_dir.exists():
        raise FileNotFoundError(f'Missing source split directory: {src_split_dir}')

    class_dirs = sorted([d for d in src_split_dir.iterdir() if d.is_dir()])
    export_items = []
    for class_dir in class_dirs:
        class_name = class_dir.name.replace('/', '_')
        for src_img in sorted(class_dir.rglob('*')):
            if src_img.is_file() and src_img.suffix.lower() in valid_ext:
                export_items.append((class_name, src_img))

    class_counts = {}
    for class_name, src_img in tqdm(export_items, desc=f'Imagenette {split_name}'):
        class_dir = dst_split_dir / class_name
        class_dir.mkdir(parents=True, exist_ok=True)

        idx = class_counts.get(class_name, 0)
        class_counts[class_name] = idx + 1
        out_file = class_dir / f'{idx:08d}.jpg'

        if out_file.exists():
            continue

        with Image.open(src_img) as im:
            save_jpeg_robust(im, out_file)

    marker_file.write_text('ok\n')
    print(f'Imagenette {split_name} export complete -> {dst_split_dir}')


## 4) Explore Imagenette

Before training, do a quick sanity check: view random samples and verify class balance. This helps understanding the dataset composition better.


In [ ]:
import matplotlib.pyplot as plt

explore_ds = datasets.ImageFolder(str(IMAGENETTE_DIR / 'train'))
class_names = explore_ds.classes
class_counts = np.bincount(explore_ds.targets, minlength=len(class_names))

synset_to_label = {
    'n01440764': 'tench',
    'n02102040': 'English springer',
    'n02979186': 'cassette player',
    'n03000684': 'chain saw',
    'n03028079': 'church',
    'n03394916': 'French horn',
    'n03417042': 'garbage truck',
    'n03425413': 'gas pump',
    'n03445777': 'golf ball',
    'n03888257': 'parachute',
}
pretty_names = [synset_to_label.get(c, c) for c in class_names]

rng = np.random.default_rng(42)
targets = np.array(explore_ds.targets)
sample_idxs = []
for class_idx in range(len(class_names)):
    class_pool = np.where(targets == class_idx)[0]
    sample_idxs.append(int(rng.choice(class_pool)))

n_cols = 5
n_rows = int(np.ceil(len(sample_idxs) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 2.8 * n_rows))
axes = np.array(axes).reshape(-1)
for ax, idx in zip(axes, sample_idxs):
    img, label = explore_ds[idx]
    ax.imshow(img)
    ax.set_title(pretty_names[label], fontsize=9)
    ax.axis('off')
for ax in axes[len(sample_idxs):]:
    ax.axis('off')
fig.suptitle('One random sample per Imagenette class')
fig.tight_layout()
plt.show()

plt.figure(figsize=(12, 3.5))
plt.bar(pretty_names, class_counts)
plt.title('Imagenette train class distribution')
plt.xlabel('Class')
plt.ylabel('Images')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

print(f'Classes: {len(class_names)} | Train images: {len(explore_ds)}')


## 5) Create Dataloaders

Build normalized 224x224 train/val dataloaders to match the original MAE setup.

In [ ]:
SEED = 42
NUM_WORKERS = 1
BATCH_SIZE = 64
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

transform_eval = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def build_imagefolder_loaders(root_dir: Path, batch_size: int):
    train_ds = datasets.ImageFolder(str(root_dir / 'train'), transform=transform_eval)
    val_ds = datasets.ImageFolder(str(root_dir / 'val'), transform=transform_eval)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    return train_ds, val_ds, train_loader, val_loader

train_ds, val_ds, train_loader, val_loader = build_imagefolder_loaders(IMAGENETTE_DIR, BATCH_SIZE)


x, y = next(iter(train_loader))
print(f'imagenette train={len(train_ds):6d} | val={len(val_ds):6d} | batch_x={tuple(x.shape)} | batch_y={tuple(y.shape)}')

## 6) MAE Model Setup

Import MAE model definitions, choose a model variant, and load the model checkpoint.


In [ ]:
if str(MAE_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(MAE_REPO_DIR))

import models_mae

MAE_MODEL_NAME = 'mae_vit_base_patch16'

available = [k for k in models_mae.__dict__.keys() if k.startswith('mae_vit_')]
if MAE_MODEL_NAME not in models_mae.__dict__:
    raise ValueError(f'MAE_MODEL_NAME={MAE_MODEL_NAME} not found. Available: {available}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models_mae.__dict__[MAE_MODEL_NAME](norm_pix_loss=False).to(device)

print(f'Model: {MAE_MODEL_NAME}')
print(f'Device: {device}')
ckpt_url = 'https://dl.fbaipublicfiles.com/mae/visualize/mae_visualize_vit_base.pth'
ckpt_name = Path(ckpt_url).name
ckpt_path = CHECKPOINTS_DIR / ckpt_name
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
if not ckpt_path.exists() and ckpt_url is not None:
    print(f'Downloading checkpoint to {ckpt_path}...')
    try:
        urllib.request.urlretrieve(ckpt_url, ckpt_path)
    except Exception as exc:
        print(f'Checkpoint download failed: {exc}')
else:
    print(f'Using existing checkpoint: {ckpt_path}')

checkpoint = torch.load(ckpt_path, map_location='cpu')
model_state = checkpoint['model']
msg = model.load_state_dict(model_state, strict=True)
missing = list(msg.missing_keys)
unexpected = list(msg.unexpected_keys)
print(f'Checkpoint loaded. missing={len(missing)}, unexpected={len(unexpected)}')


## 7) Visualize MAE Reconstruction

Run the pretrained MAE on a validation batch with a fixed mask ratio. Compare original, masked input, reconstructed image, and raw decoder output.


In [ ]:
import matplotlib.pyplot as plt


MASK_RATIO = 0.75
mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
std = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)


def denorm(x):
    return (x * std + mean).clamp(0, 1)


rng = np.random.default_rng()  # fresh entropy each run for new samples
n_take = min(BATCH_SIZE, len(val_ds))
idxs = rng.choice(len(val_ds), size=n_take, replace=False).tolist()
images = torch.stack([val_ds[i][0] for i in idxs], dim=0).to(device)

with torch.no_grad():
    loss, pred, mask = model(images, mask_ratio=MASK_RATIO)

patch_size = model.patch_embed.patch_size[0]
mask = mask.detach()
pred = pred.detach()

mask_tokens = mask.unsqueeze(-1).repeat(1, 1, patch_size * patch_size * 3)
mask_img = model.unpatchify(mask_tokens)
pred_img = model.unpatchify(pred)

img_masked = images * (1 - mask_img)
img_paste = images * (1 - mask_img) + pred_img * mask_img

orig_vis = denorm(images).cpu()
masked_vis = denorm(img_masked).cpu()
recon_vis = denorm(img_paste).cpu()
pred_only_vis = denorm(pred_img).cpu()

n_show = min(8, orig_vis.shape[0])
fig, axes = plt.subplots(n_show, 4, figsize=(12, 2.6 * n_show))
if n_show == 1:
    axes = np.expand_dims(axes, axis=0)

titles = ['original', 'masked', 'reconstruction + visible', 'reconstruction_only']
for c, t in enumerate(titles):
    axes[0, c].set_title(t)

for i in range(n_show):
    panels = [orig_vis[i], masked_vis[i], recon_vis[i], pred_only_vis[i]]
    for j, panel in enumerate(panels):
        axes[i, j].imshow(panel.permute(1, 2, 0).numpy())
        axes[i, j].axis('off')

fig.tight_layout()
out_path = RECON_DIR / f'imagenette_mask{MASK_RATIO:.2f}.png'
fig.savefig(out_path, dpi=160, bbox_inches='tight')
print(f'Saved reconstruction figure: {out_path}')
print(f'imagenette batch MAE loss: {float(loss):.4f}')
plt.show()


## 8) Mini Pretraining Run

Run a short pretraining loop on a tiny fixed batch for fast feedback. Loss is printed during training, and model checkpoints are saved at selected target steps. The next cell visualizes reconstruction progress from those saved checkpoints.


In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
    print("WARNING: no GPU detected, using CPU for pretraining. Run will be slow.")

base_model = models_mae.__dict__['mae_vit_base_patch16'](norm_pix_loss=False).to(device)
pretrain_loader = DataLoader(
        train_ds,
        batch_size=4,
        shuffle=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

# Use one fixed tiny batch so progress is visible quickly.
fixed_images, _ = next(iter(pretrain_loader))
fixed_images = fixed_images.to(device)
progress_steps = [0, 200, 400]
log_every = 20

optimizer = torch.optim.AdamW(base_model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.05)
train_losses = []

mini_pretrain_dir = CHECKPOINTS_DIR / 'mini_pretrain'
mini_pretrain_dir.mkdir(parents=True, exist_ok=True)
pretrain_step_ckpts = {}


def save_step_ckpt(step):
    ckpt_path = mini_pretrain_dir / f'mae_pretrain_step_{step:04d}.pth'
    torch.save(base_model.state_dict(), ckpt_path)
    pretrain_step_ckpts[step] = ckpt_path


base_model.eval()
save_step_ckpt(0)
base_model.train()

for step in range(1, progress_steps[-1] + 1):
    loss, _, _ = base_model(fixed_images, mask_ratio=MASK_RATIO)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    current_loss = float(loss.item())
    train_losses.append(current_loss)

    if step == 1 or step % log_every == 0 or step in progress_steps:
        print(f'[pretrain] step {step:>3}/{progress_steps[-1]} loss={current_loss:.4f}')

    if step in progress_steps:
        save_step_ckpt(step)

pretrain_ckpt = CHECKPOINTS_DIR / 'mae_pretrain_last.pth'
torch.save({
    'model': base_model.state_dict(),
    'progress_steps': progress_steps,
    'train_losses': train_losses,
    'step_checkpoints': {k: str(v) for k, v in pretrain_step_ckpts.items()},
}, pretrain_ckpt)

print(f'Saved final mini-pretrain checkpoint: {pretrain_ckpt}')
print(f'Train loss first={train_losses[0]:.4f}, last={train_losses[-1]:.4f}')


In [ ]:
# Visualize multiple samples across saved pretraining checkpoints.
row_steps = sorted(pretrain_step_ckpts.keys())
n_show_samples = min(4, fixed_images.shape[0])

viz_model = models_mae.__dict__['mae_vit_base_patch16'](norm_pix_loss=False).to(device)
group_gap = 1
rows = len(row_steps) * n_show_samples + group_gap * max(0, len(row_steps) - 1)
fig, axes = plt.subplots(rows, 4, figsize=(12, 2.6 * rows))
if rows == 1:
    axes = np.expand_dims(axes, axis=0)

for r in range(rows):
    for c in range(4):
        axes[r, c].axis('off')

titles = ['original', 'masked', 'reconstruction + visible', 'reconstruction_only']
for c, t in enumerate(titles):
    axes[0, c].set_title(t)

row_idx = 0
separator_rows = []
for step in row_steps:
    state_dict = torch.load(pretrain_step_ckpts[step], map_location='cpu')
    viz_model.load_state_dict(state_dict, strict=True)
    viz_model.eval()

    with torch.no_grad():
        loss, pred, mask = viz_model(fixed_images, mask_ratio=MASK_RATIO)

    p = viz_model.patch_embed.patch_size[0]
    mask_tokens = mask.unsqueeze(-1).repeat(1, 1, p * p * 3)
    mask_img = viz_model.unpatchify(mask_tokens)
    pred_img = viz_model.unpatchify(pred)

    img_masked = fixed_images * (1 - mask_img)
    img_paste = fixed_images * (1 - mask_img) + pred_img * mask_img

    orig_vis = denorm(fixed_images).cpu()
    masked_vis = denorm(img_masked).cpu()
    recon_vis = denorm(img_paste).cpu()
    pred_only_vis = denorm(pred_img).cpu()

    step_loss = float(loss.item())
    for sample_idx in range(n_show_samples):
        panels = [
            orig_vis[sample_idx],
            masked_vis[sample_idx],
            recon_vis[sample_idx],
            pred_only_vis[sample_idx],
        ]
        for c, panel in enumerate(panels):
            axes[row_idx, c].imshow(panel.permute(1, 2, 0).numpy())
            axes[row_idx, c].axis('off')
        axes[row_idx, 0].set_ylabel(
            f'step {step}\nsample {sample_idx}\nloss {step_loss:.4f}',
            rotation=0,
            labelpad=52,
            va='center',
        )
        row_idx += 1

    if step != row_steps[-1]:
        separator_rows.append(row_idx)
        row_idx += group_gap

fig.tight_layout()
for sep_row in separator_rows:
    y = axes[sep_row, 0].get_position().y0 + 0.5 * axes[sep_row, 0].get_position().height
    fig.add_artist(plt.Line2D([0.05, 0.98], [y, y], transform=fig.transFigure, color='black', lw=0.8, alpha=0.35))

pretrain_fig = RECON_DIR / f'pretrain_progress_samples{n_show_samples}_mask{MASK_RATIO:.2f}.png'
fig.savefig(pretrain_fig, dpi=160, bbox_inches='tight')
print(f'Saved pretrain progress figure: {pretrain_fig}')
plt.show()
